# Network Traffic Behavior Clustering

Discover common flow behaviors and isolate unusual scan-like traffic without using labels during training.

**Safety and scope:** This project uses synthetic, non-sensitive telemetry for defensive analytics. It does not perform exploitation or execute malicious content.

## Goal

Implement K-means from scratch, profile each cluster, and measure cluster purity against hidden synthetic labels.


## Setup

The notebook is deterministic, runs offline, and implements the core analytical method directly with NumPy and Pandas so the modeling logic remains inspectable.


In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 120)

def kmeans(values, clusters, iterations=80):
    centroids = values[rng.choice(len(values), clusters, replace=False)].copy()
    labels = np.zeros(len(values), dtype=int)
    for _ in range(iterations):
        distances = ((values[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        new_labels = distances.argmin(axis=1)
        new_centroids = np.vstack([
            values[new_labels == cluster].mean(axis=0)
            if np.any(new_labels == cluster) else centroids[cluster]
            for cluster in range(clusters)
        ])
        if np.allclose(new_centroids, centroids, atol=1e-5):
            labels = new_labels
            break
        centroids = new_centroids
        labels = new_labels
    return labels, centroids


## Steps

### 1. Generate synthetic network flows


In [2]:
behavior_specs = {
    "web": ([5.0, 7.8, 2.0, 0.08], [0.5, 0.6, 0.4, 0.03], 260),
    "dns": ([3.2, 3.8, 1.2, 0.02], [0.35, 0.45, 0.25, 0.02], 180),
    "file_transfer": ([7.0, 10.2, 4.1, 0.14], [0.55, 0.7, 0.45, 0.05], 130),
    "port_scan": ([1.7, 2.0, 6.2, 0.78], [0.30, 0.35, 0.50, 0.08], 90),
}

flow_parts = []
for behavior, (means, scales, count) in behavior_specs.items():
    values = rng.normal(means, scales, size=(count, 4))
    part = pd.DataFrame(values, columns=["log_packets", "log_bytes", "log_destinations", "syn_ratio"])
    part["behavior"] = behavior
    flow_parts.append(part)

flows = pd.concat(flow_parts, ignore_index=True)
flows["syn_ratio"] = flows["syn_ratio"].clip(0, 1)
print("Flow count:", len(flows))
print(flows.groupby("behavior").size().to_string())
print("\nSample flows:")
print(flows.sample(6, random_state=SEED).round(3).to_string(index=False))


Flow count: 660
behavior
dns              180
file_transfer    130
port_scan         90
web              260

Sample flows:
 log_packets  log_bytes  log_destinations  syn_ratio      behavior
       1.703      2.503             6.478      0.643     port_scan
       7.167     10.136             3.861      0.251 file_transfer
       4.157      7.304             2.099      0.075           web
       6.398     10.919             3.235      0.196 file_transfer
       4.924      8.030             2.400      0.048           web
       6.516     10.672             4.487      0.218 file_transfer


### 2. Cluster and profile behaviors


In [3]:
feature_names = ["log_packets", "log_bytes", "log_destinations", "syn_ratio"]
raw_features = flows[feature_names].to_numpy(float)
scaled_features = (raw_features - raw_features.mean(axis=0)) / raw_features.std(axis=0)
cluster_label, centroids = kmeans(scaled_features, clusters=4)
flows["cluster"] = cluster_label

cluster_profiles = flows.groupby("cluster")[feature_names].mean().round(3)
contingency = pd.crosstab(flows["cluster"], flows["behavior"])
majority_correct = contingency.max(axis=1).sum()
cluster_purity = majority_correct / len(flows)

print("Cluster profiles:")
print(cluster_profiles.to_string())
print("\nCluster-to-hidden-label comparison:")
print(contingency.to_string())
print("\nCluster purity:", round(cluster_purity, 3))


Cluster profiles:
         log_packets  log_bytes  log_destinations  syn_ratio
cluster                                                     
0              6.892     10.240             4.091      0.128
1              1.741      2.015             6.197      0.795
2              3.209      3.756             1.159      0.022
3              4.974      7.775             2.001      0.079

Cluster-to-hidden-label comparison:
behavior  dns  file_transfer  port_scan  web
cluster                                     
0           0            130          0    0
1           0              0         90    0
2         180              0          0    0
3           0              0          0  260

Cluster purity: 1.0


## Checks


In [4]:
assert len(np.unique(cluster_label)) == 4
assert cluster_purity >= 0.85
assert not cluster_profiles.isna().any().any()
scan_cluster = contingency["port_scan"].idxmax()
assert cluster_profiles.loc[scan_cluster, "syn_ratio"] > 0.5
print("Checks passed: four stable clusters, strong purity, and a distinct scan-like cluster.")


Checks passed: four stable clusters, strong purity, and a distinct scan-like cluster.


## Next Steps

        - Derive flow features from Zeek, NetFlow, or firewall telemetry.
- Track cluster drift by week to detect emerging behaviors.
- Compare K-means with density-based clustering for irregular traffic shapes.
